# 🚀 Stock Price Prediction - Advanced LSTM with Attention

**Modelo Mejorado con:**
- ✅ Feature Engineering Avanzado (OHLCV + Indicadores Técnicos)
- ✅ Bidirectional LSTM con mecanismo de Attention
- ✅ Regularización (Dropout, BatchNorm)
- ✅ Callbacks avanzados (Early Stopping, ReduceLROnPlateau)
- ✅ Predicción Multi-Horizonte (días, semanas, meses, años)
- ✅ Intervalos de Confianza
- ✅ Evaluación Robusta

# 1. Importación de Librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Dense, LSTM, Dropout, Bidirectional, Input,
    BatchNormalization, Attention, Concatenate,
    Layer, MultiHeadAttention, LayerNormalization,
    GlobalAveragePooling1D
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

# Preprocessing & Metrics
from sklearn.preprocessing import MinMaxScaler, RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

# 2. Configuración Global

In [ ]:
# ============= CONFIGURACIÓN =============
CONFIG = {
    'ticker': 'AAPL',           # Símbolo de la acción
    'years_history': 10,         # Años de datos históricos
    'sequence_length': 60,       # Días de secuencia (ventana temporal)
    'train_split': 0.8,          # 80% entrenamiento
    'val_split': 0.1,            # 10% validación
    'test_split': 0.1,           # 10% testing
    'batch_size': 32,
    'epochs': 100,
    'learning_rate': 0.001,
    'lstm_units': [128, 64],     # Unidades por capa LSTM
    'dropout_rate': 0.2,
    'attention_heads': 4,
    'patience': 15,              # Early stopping patience
}

# Semilla para reproducibilidad
np.random.seed(42)
tf.random.set_seed(42)

# 3. Descarga y Exploración de Datos

In [ ]:
# Descargar datos
now = datetime.now()
start = datetime(now.year - CONFIG['years_history'], now.month, now.day)
end = now

print(f"Descargando datos de {CONFIG['ticker']} desde {start.date()} hasta {end.date()}...")
df = yf.download(CONFIG['ticker'], start, end, progress=False)

# Limpiar columnas si es MultiIndex
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

df = df.reset_index()
print(f"\n📊 Datos descargados: {len(df)} registros")
print(f"   Rango: {df['Date'].min()} - {df['Date'].max()}")
df.head()

In [ ]:
# Estadísticas descriptivas
print("📈 Estadísticas Descriptivas:")
df.describe()

In [ ]:
# Verificar valores nulos
print("🔍 Valores Nulos por Columna:")
print(df.isna().sum())

# Rellenar valores nulos si existen
df = df.fillna(method='ffill').fillna(method='bfill')

# 4. Feature Engineering Avanzado 🔧

In [ ]:
def calculate_technical_indicators(df):
    """
    Calcula indicadores técnicos avanzados para el análisis de acciones.
    """
    data = df.copy()
    
    # ===== MEDIAS MÓVILES =====
    for window in [7, 14, 21, 50, 100, 200]:
        data[f'SMA_{window}'] = data['Close'].rolling(window=window).mean()
        data[f'EMA_{window}'] = data['Close'].ewm(span=window, adjust=False).mean()
    
    # ===== RSI (Relative Strength Index) =====
    delta = data['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    data['RSI'] = 100 - (100 / (1 + rs))
    
    # ===== MACD =====
    ema_12 = data['Close'].ewm(span=12, adjust=False).mean()
    ema_26 = data['Close'].ewm(span=26, adjust=False).mean()
    data['MACD'] = ema_12 - ema_26
    data['MACD_Signal'] = data['MACD'].ewm(span=9, adjust=False).mean()
    data['MACD_Hist'] = data['MACD'] - data['MACD_Signal']
    
    # ===== BOLLINGER BANDS =====
    sma_20 = data['Close'].rolling(window=20).mean()
    std_20 = data['Close'].rolling(window=20).std()
    data['BB_Upper'] = sma_20 + (std_20 * 2)
    data['BB_Lower'] = sma_20 - (std_20 * 2)
    data['BB_Width'] = (data['BB_Upper'] - data['BB_Lower']) / sma_20
    data['BB_Position'] = (data['Close'] - data['BB_Lower']) / (data['BB_Upper'] - data['BB_Lower'])
    
    # ===== ATR (Average True Range) - Volatilidad =====
    high_low = data['High'] - data['Low']
    high_close = np.abs(data['High'] - data['Close'].shift())
    low_close = np.abs(data['Low'] - data['Close'].shift())
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    data['ATR'] = tr.rolling(window=14).mean()
    
    # ===== VOLATILIDAD HISTÓRICA =====
    data['Returns'] = data['Close'].pct_change()
    data['Log_Returns'] = np.log(data['Close'] / data['Close'].shift(1))
    data['Volatility_21'] = data['Returns'].rolling(window=21).std() * np.sqrt(252)  # Anualizada
    
    # ===== MOMENTUM =====
    data['Momentum_10'] = data['Close'] - data['Close'].shift(10)
    data['Momentum_20'] = data['Close'] - data['Close'].shift(20)
    data['ROC'] = ((data['Close'] - data['Close'].shift(10)) / data['Close'].shift(10)) * 100
    
    # ===== VOLUME INDICATORS =====
    data['Volume_SMA_20'] = data['Volume'].rolling(window=20).mean()
    data['Volume_Ratio'] = data['Volume'] / data['Volume_SMA_20']
    
    # ===== PRICE CHANNELS =====
    data['High_20'] = data['High'].rolling(window=20).max()
    data['Low_20'] = data['Low'].rolling(window=20).min()
    data['Price_Position'] = (data['Close'] - data['Low_20']) / (data['High_20'] - data['Low_20'])
    
    # ===== STOCHASTIC OSCILLATOR =====
    low_14 = data['Low'].rolling(window=14).min()
    high_14 = data['High'].rolling(window=14).max()
    data['Stoch_K'] = 100 * ((data['Close'] - low_14) / (high_14 - low_14))
    data['Stoch_D'] = data['Stoch_K'].rolling(window=3).mean()
    
    # ===== TEMPORAL FEATURES =====
    data['DayOfWeek'] = pd.to_datetime(data['Date']).dt.dayofweek / 6  # Normalizado 0-1
    data['Month'] = pd.to_datetime(data['Date']).dt.month / 12  # Normalizado 0-1
    data['Quarter'] = pd.to_datetime(data['Date']).dt.quarter / 4  # Normalizado 0-1
    data['DayOfYear'] = pd.to_datetime(data['Date']).dt.dayofyear / 365  # Normalizado 0-1
    
    return data

# Aplicar feature engineering
df_features = calculate_technical_indicators(df)
print(f"\n🔧 Features generados: {len(df_features.columns)} columnas")
df_features.columns.tolist()

In [ ]:
# Eliminar filas con NaN debido a los indicadores técnicos
df_clean = df_features.dropna().reset_index(drop=True)
print(f"\n📊 Datos después de limpieza: {len(df_clean)} registros (eliminados {len(df_features) - len(df_clean)} por NaN)")
df_clean.tail()

# 5. Visualización de Features

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 16))

# Precio con Medias Móviles
axes[0].plot(df_clean['Date'], df_clean['Close'], label='Close', linewidth=1)
axes[0].plot(df_clean['Date'], df_clean['SMA_50'], label='SMA 50', alpha=0.7)
axes[0].plot(df_clean['Date'], df_clean['SMA_200'], label='SMA 200', alpha=0.7)
axes[0].fill_between(df_clean['Date'], df_clean['BB_Lower'], df_clean['BB_Upper'], alpha=0.2, label='Bollinger Bands')
axes[0].set_title(f'{CONFIG["ticker"]} - Precio con Indicadores')
axes[0].legend(loc='upper left')
axes[0].grid(True, alpha=0.3)

# RSI
axes[1].plot(df_clean['Date'], df_clean['RSI'], color='purple', linewidth=1)
axes[1].axhline(y=70, color='r', linestyle='--', alpha=0.5)
axes[1].axhline(y=30, color='g', linestyle='--', alpha=0.5)
axes[1].fill_between(df_clean['Date'], 30, 70, alpha=0.1)
axes[1].set_title('RSI (14)')
axes[1].set_ylim(0, 100)
axes[1].grid(True, alpha=0.3)

# MACD
axes[2].plot(df_clean['Date'], df_clean['MACD'], label='MACD', linewidth=1)
axes[2].plot(df_clean['Date'], df_clean['MACD_Signal'], label='Signal', linewidth=1)
axes[2].bar(df_clean['Date'], df_clean['MACD_Hist'], alpha=0.3, label='Histogram')
axes[2].axhline(y=0, color='black', linestyle='-', alpha=0.3)
axes[2].set_title('MACD')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

# Volatilidad
axes[3].plot(df_clean['Date'], df_clean['Volatility_21'], color='orange', linewidth=1)
axes[3].set_title('Volatilidad Anualizada (21 días)')
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 6. Preparación de Datos para el Modelo

In [ ]:
# Seleccionar features para el modelo
FEATURE_COLUMNS = [
    'Open', 'High', 'Low', 'Close', 'Volume',
    'SMA_7', 'SMA_14', 'SMA_21', 'SMA_50',
    'EMA_7', 'EMA_14', 'EMA_21',
    'RSI', 'MACD', 'MACD_Signal', 'MACD_Hist',
    'BB_Width', 'BB_Position',
    'ATR', 'Volatility_21',
    'Momentum_10', 'ROC',
    'Volume_Ratio', 'Price_Position',
    'Stoch_K', 'Stoch_D',
    'DayOfWeek', 'Month'
]

TARGET_COLUMN = 'Close'

print(f"📊 Features seleccionados: {len(FEATURE_COLUMNS)}")
print(FEATURE_COLUMNS)

In [ ]:
# Extraer datos
dates = df_clean['Date'].values
feature_data = df_clean[FEATURE_COLUMNS].values
target_data = df_clean[TARGET_COLUMN].values.reshape(-1, 1)

print(f"Shape features: {feature_data.shape}")
print(f"Shape target: {target_data.shape}")

In [ ]:
# Normalización con RobustScaler (más robusto a outliers)
feature_scaler = RobustScaler()
target_scaler = MinMaxScaler(feature_range=(0, 1))

# Fit en datos de entrenamiento solamente (evitar data leakage)
train_size = int(len(feature_data) * CONFIG['train_split'])

feature_scaler.fit(feature_data[:train_size])
target_scaler.fit(target_data[:train_size])

# Transform all data
scaled_features = feature_scaler.transform(feature_data)
scaled_target = target_scaler.transform(target_data)

print(f"✅ Datos normalizados")
print(f"   Features range: [{scaled_features.min():.4f}, {scaled_features.max():.4f}]")
print(f"   Target range: [{scaled_target.min():.4f}, {scaled_target.max():.4f}]")

In [ ]:
def create_sequences(features, target, sequence_length):
    """
    Crea secuencias para entrenamiento del modelo LSTM.
    X: secuencias de features
    y: valor objetivo siguiente
    """
    X, y = [], []
    for i in range(sequence_length, len(features)):
        X.append(features[i-sequence_length:i])
        y.append(target[i, 0])
    return np.array(X), np.array(y)

# Crear secuencias
X, y = create_sequences(scaled_features, scaled_target, CONFIG['sequence_length'])

# Ajustar fechas (perdemos las primeras por la ventana)
dates_seq = dates[CONFIG['sequence_length']:]

print(f"\n📊 Secuencias creadas:")
print(f"   X shape: {X.shape} (samples, timesteps, features)")
print(f"   y shape: {y.shape}")

In [ ]:
# División Train/Validation/Test
total_samples = len(X)
train_end = int(total_samples * CONFIG['train_split'])
val_end = int(total_samples * (CONFIG['train_split'] + CONFIG['val_split']))

X_train, y_train = X[:train_end], y[:train_end]
X_val, y_val = X[train_end:val_end], y[train_end:val_end]
X_test, y_test = X[val_end:], y[val_end:]

dates_train = dates_seq[:train_end]
dates_val = dates_seq[train_end:val_end]
dates_test = dates_seq[val_end:]

print(f"\n📊 División de datos:")
print(f"   Train: {len(X_train)} samples ({CONFIG['train_split']*100:.0f}%)")
print(f"   Val:   {len(X_val)} samples ({CONFIG['val_split']*100:.0f}%)")
print(f"   Test:  {len(X_test)} samples ({CONFIG['test_split']*100:.0f}%)")

# 7. Arquitectura del Modelo: Bidirectional LSTM + Attention 🧠

In [ ]:
class AttentionLayer(Layer):
    """
    Capa de atención personalizada para secuencias temporales.
    Permite al modelo enfocarse en los timesteps más relevantes.
    """
    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)
        
    def build(self, input_shape):
        self.W = self.add_weight(
            name='attention_weight',
            shape=(input_shape[-1], 1),
            initializer='glorot_uniform',
            trainable=True
        )
        self.b = self.add_weight(
            name='attention_bias',
            shape=(input_shape[1], 1),
            initializer='zeros',
            trainable=True
        )
        super(AttentionLayer, self).build(input_shape)
        
    def call(self, x):
        # x shape: (batch_size, timesteps, features)
        e = tf.keras.backend.tanh(tf.keras.backend.dot(x, self.W) + self.b)
        a = tf.keras.backend.softmax(e, axis=1)
        output = x * a
        return tf.keras.backend.sum(output, axis=1)
    
    def get_config(self):
        return super(AttentionLayer, self).get_config()

In [ ]:
def build_advanced_lstm_model(input_shape, config):
    """
    Construye modelo avanzado con Bidirectional LSTM y Attention.
    
    Arquitectura:
    1. Bidirectional LSTM (captura patrones forward y backward)
    2. Multi-Head Attention (enfoca en timesteps importantes)
    3. LSTM adicional para captura de patrones secuenciales
    4. Dense layers con regularización
    """
    inputs = Input(shape=input_shape, name='input_layer')
    
    # Primera capa Bidirectional LSTM
    x = Bidirectional(
        LSTM(
            config['lstm_units'][0],
            return_sequences=True,
            kernel_regularizer=l2(0.001)
        ),
        name='bidirectional_lstm_1'
    )(inputs)
    x = BatchNormalization(name='batch_norm_1')(x)
    x = Dropout(config['dropout_rate'], name='dropout_1')(x)
    
    # Multi-Head Attention
    attention_output = MultiHeadAttention(
        num_heads=config['attention_heads'],
        key_dim=32,
        name='multi_head_attention'
    )(x, x)
    x = LayerNormalization(name='layer_norm_1')(x + attention_output)  # Residual connection
    
    # Segunda capa Bidirectional LSTM
    x = Bidirectional(
        LSTM(
            config['lstm_units'][1],
            return_sequences=True,
            kernel_regularizer=l2(0.001)
        ),
        name='bidirectional_lstm_2'
    )(x)
    x = BatchNormalization(name='batch_norm_2')(x)
    x = Dropout(config['dropout_rate'], name='dropout_2')(x)
    
    # Custom Attention Layer
    x = AttentionLayer(name='custom_attention')(x)
    
    # Dense layers
    x = Dense(64, activation='relu', kernel_regularizer=l2(0.001), name='dense_1')(x)
    x = Dropout(config['dropout_rate'], name='dropout_3')(x)
    x = Dense(32, activation='relu', name='dense_2')(x)
    
    # Output layer
    outputs = Dense(1, activation='linear', name='output')(x)
    
    model = Model(inputs=inputs, outputs=outputs, name='Advanced_LSTM_Attention')
    
    return model

# Construir modelo
input_shape = (CONFIG['sequence_length'], len(FEATURE_COLUMNS))
model = build_advanced_lstm_model(input_shape, CONFIG)

# Compilar modelo
model.compile(
    optimizer=Adam(learning_rate=CONFIG['learning_rate']),
    loss='huber',  # Más robusto a outliers que MSE
    metrics=['mae']
)

model.summary()

In [ ]:
# Visualizar arquitectura
tf.keras.utils.plot_model(
    model,
    to_file='model_architecture.png',
    show_shapes=True,
    show_layer_names=True,
    rankdir='TB',
    expand_nested=True,
    dpi=150
)

from IPython.display import Image
Image('model_architecture.png')

# 8. Entrenamiento del Modelo 🏋️

In [ ]:
# Callbacks
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=CONFIG['patience'],
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        'best_model.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )
]

print("🏋️ Iniciando entrenamiento...")
print(f"   Épocas máximas: {CONFIG['epochs']}")
print(f"   Batch size: {CONFIG['batch_size']}")
print(f"   Early stopping patience: {CONFIG['patience']}")

In [ ]:
# Entrenar
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=CONFIG['epochs'],
    batch_size=CONFIG['batch_size'],
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# Visualizar entrenamiento
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Validation Loss')
axes[0].set_title('Model Loss (Huber)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE
axes[1].plot(history.history['mae'], label='Train MAE')
axes[1].plot(history.history['val_mae'], label='Validation MAE')
axes[1].set_title('Mean Absolute Error')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n✅ Entrenamiento completado en {len(history.history['loss'])} épocas")
print(f"   Mejor val_loss: {min(history.history['val_loss']):.6f}")

# 9. Evaluación del Modelo 📊

In [ ]:
# Cargar mejor modelo
from tensorflow.keras.models import load_model

# Registrar capa personalizada
best_model = load_model('best_model.keras', custom_objects={'AttentionLayer': AttentionLayer})

# Predicciones en datos de test
y_pred_scaled = best_model.predict(X_test, verbose=0)

# Invertir escalado
y_pred = target_scaler.inverse_transform(y_pred_scaled).flatten()
y_true = target_scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

print(f"\n📊 Predicciones realizadas: {len(y_pred)} samples")

In [ ]:
def calculate_metrics(y_true, y_pred):
    """Calcula métricas de evaluación completas."""
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2 = r2_score(y_true, y_pred)
    
    # Dirección accuracy (si predice correctamente la dirección del movimiento)
    direction_true = np.diff(y_true) > 0
    direction_pred = np.diff(y_pred) > 0
    direction_accuracy = np.mean(direction_true == direction_pred) * 100
    
    return {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'R²': r2,
        'Direction Accuracy': direction_accuracy
    }

metrics = calculate_metrics(y_true, y_pred)

print("\n" + "="*50)
print("📈 MÉTRICAS DE EVALUACIÓN")
print("="*50)
for metric_name, value in metrics.items():
    if metric_name in ['MAPE', 'Direction Accuracy', 'R²']:
        print(f"   {metric_name}: {value:.2f}{'%' if metric_name != 'R²' else ''}")
    else:
        print(f"   {metric_name}: ${value:.2f}")
print("="*50)

In [ ]:
# Visualización de predicciones vs real
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Gráfico completo
axes[0].plot(dates_test, y_true, 'b-', label='Precio Real', linewidth=1.5, alpha=0.8)
axes[0].plot(dates_test, y_pred, 'r-', label='Predicción', linewidth=1.5, alpha=0.8)
axes[0].fill_between(dates_test, y_true, y_pred, alpha=0.2, color='gray')
axes[0].set_title(f'{CONFIG["ticker"]} - Predicción vs Real (Test Set)', fontsize=14)
axes[0].set_xlabel('Fecha')
axes[0].set_ylabel('Precio ($)')
axes[0].legend(loc='upper left')
axes[0].grid(True, alpha=0.3)

# Scatter plot
axes[1].scatter(y_true, y_pred, alpha=0.5, edgecolors='k', linewidth=0.5)
axes[1].plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', label='Predicción Perfecta')
axes[1].set_title(f'Scatter Plot: Real vs Predicción (R² = {metrics["R²"]:.4f})', fontsize=14)
axes[1].set_xlabel('Precio Real ($)')
axes[1].set_ylabel('Precio Predicho ($)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Análisis de errores
errors = y_true - y_pred
percentage_errors = (errors / y_true) * 100

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribución de errores
axes[0].hist(errors, bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(x=0, color='r', linestyle='--')
axes[0].set_title('Distribución de Errores ($)')
axes[0].set_xlabel('Error')
axes[0].set_ylabel('Frecuencia')

# Errores porcentuales
axes[1].hist(percentage_errors, bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[1].axvline(x=0, color='r', linestyle='--')
axes[1].set_title('Distribución de Errores (%)')
axes[1].set_xlabel('Error %')
axes[1].set_ylabel('Frecuencia')

# Errores en el tiempo
axes[2].plot(dates_test, errors, alpha=0.7)
axes[2].axhline(y=0, color='r', linestyle='--')
axes[2].fill_between(dates_test, 0, errors, alpha=0.3)
axes[2].set_title('Errores a lo largo del tiempo')
axes[2].set_xlabel('Fecha')
axes[2].set_ylabel('Error ($)')

plt.tight_layout()
plt.show()

print(f"\n📊 Estadísticas de Error:")
print(f"   Error promedio: ${np.mean(errors):.2f}")
print(f"   Error std: ${np.std(errors):.2f}")
print(f"   Error min: ${np.min(errors):.2f}")
print(f"   Error max: ${np.max(errors):.2f}")

# 10. 🔮 Predicción Futura Multi-Horizonte

In [ ]:
class FuturePredictor:
    """
    Clase para realizar predicciones futuras con el modelo entrenado.
    Soporta predicciones por días, semanas, meses y años.
    """
    
    def __init__(self, model, feature_scaler, target_scaler, feature_columns, sequence_length):
        self.model = model
        self.feature_scaler = feature_scaler
        self.target_scaler = target_scaler
        self.feature_columns = feature_columns
        self.sequence_length = sequence_length
        
    def _update_features_for_prediction(self, last_sequence, predicted_price, current_date):
        """
        Actualiza los features para la siguiente predicción.
        Simula los indicadores técnicos basándose en el precio predicho.
        """
        # Invertir el escalado de la última secuencia
        last_seq_original = self.feature_scaler.inverse_transform(last_sequence)
        
        # Crear nueva fila con precio predicho
        new_row = last_seq_original[-1].copy()
        
        # Actualizar OHLC (simplificación: usamos el precio predicho para todos)
        close_idx = self.feature_columns.index('Close')
        open_idx = self.feature_columns.index('Open')
        high_idx = self.feature_columns.index('High')
        low_idx = self.feature_columns.index('Low')
        
        prev_close = last_seq_original[-1, close_idx]
        
        # Simular OHLC con volatilidad
        volatility = abs(predicted_price - prev_close) * 0.5
        new_row[open_idx] = prev_close
        new_row[close_idx] = predicted_price
        new_row[high_idx] = max(prev_close, predicted_price) + volatility * 0.3
        new_row[low_idx] = min(prev_close, predicted_price) - volatility * 0.3
        
        # Actualizar medias móviles (aproximación)
        for col in self.feature_columns:
            if col.startswith('SMA_') or col.startswith('EMA_'):
                window = int(col.split('_')[1])
                col_idx = self.feature_columns.index(col)
                # Aproximación: media ponderada con el nuevo precio
                new_row[col_idx] = (last_seq_original[-1, col_idx] * (window-1) + predicted_price) / window
        
        # Actualizar features temporales
        if 'DayOfWeek' in self.feature_columns:
            dow_idx = self.feature_columns.index('DayOfWeek')
            new_row[dow_idx] = current_date.weekday() / 6
        if 'Month' in self.feature_columns:
            month_idx = self.feature_columns.index('Month')
            new_row[month_idx] = current_date.month / 12
        
        # Escalar la nueva fila
        new_row_scaled = self.feature_scaler.transform(new_row.reshape(1, -1))
        
        # Actualizar secuencia (shift y agregar nueva fila)
        new_sequence = np.vstack([last_sequence[1:], new_row_scaled])
        
        return new_sequence
    
    def predict_future(self, last_data, days=None, weeks=None, months=None, years=None, 
                       target_date=None, confidence_intervals=True, n_simulations=100):
        """
        Predice precios futuros.
        
        Args:
            last_data: DataFrame con los últimos datos históricos
            days: Número de días a predecir
            weeks: Número de semanas a predecir
            months: Número de meses a predecir  
            years: Número de años a predecir
            target_date: Fecha específica hasta la cual predecir (datetime)
            confidence_intervals: Si calcular intervalos de confianza
            n_simulations: Número de simulaciones para intervalos de confianza
            
        Returns:
            DataFrame con predicciones y opcionalmente intervalos de confianza
        """
        # Calcular días de trading a predecir
        if target_date is not None:
            last_date = pd.to_datetime(last_data['Date'].iloc[-1])
            target_date = pd.to_datetime(target_date)
            total_days = (target_date - last_date).days
            trading_days = int(total_days * 252 / 365)  # Aprox días de trading
        else:
            trading_days = 0
            if days:
                trading_days += days
            if weeks:
                trading_days += weeks * 5  # 5 días de trading por semana
            if months:
                trading_days += months * 21  # ~21 días de trading por mes
            if years:
                trading_days += years * 252  # ~252 días de trading por año
        
        if trading_days <= 0:
            raise ValueError("Debe especificar al menos un horizonte de predicción")
        
        print(f"\n🔮 Prediciendo {trading_days} días de trading...")
        
        # Preparar datos iniciales
        last_features = last_data[self.feature_columns].values[-self.sequence_length:]
        last_sequence = self.feature_scaler.transform(last_features)
        
        last_date = pd.to_datetime(last_data['Date'].iloc[-1])
        
        if confidence_intervals:
            all_predictions = []
            
            for sim in range(n_simulations):
                predictions = []
                current_sequence = last_sequence.copy()
                current_date = last_date
                
                for day in range(trading_days):
                    # Predecir siguiente día
                    pred_input = current_sequence.reshape(1, self.sequence_length, -1)
                    pred_scaled = self.model.predict(pred_input, verbose=0)[0, 0]
                    
                    # Agregar ruido para simular incertidumbre
                    noise = np.random.normal(0, 0.02) if sim > 0 else 0
                    pred_scaled_noisy = pred_scaled * (1 + noise)
                    
                    # Invertir escalado
                    pred_price = self.target_scaler.inverse_transform([[pred_scaled_noisy]])[0, 0]
                    predictions.append(pred_price)
                    
                    # Actualizar secuencia para siguiente predicción
                    current_date += timedelta(days=1)
                    while current_date.weekday() >= 5:  # Saltar fines de semana
                        current_date += timedelta(days=1)
                    
                    current_sequence = self._update_features_for_prediction(
                        current_sequence, pred_price, current_date
                    )
                
                all_predictions.append(predictions)
            
            all_predictions = np.array(all_predictions)
            mean_predictions = np.mean(all_predictions, axis=0)
            lower_bound = np.percentile(all_predictions, 5, axis=0)
            upper_bound = np.percentile(all_predictions, 95, axis=0)
            
        else:
            predictions = []
            current_sequence = last_sequence.copy()
            current_date = last_date
            
            for day in range(trading_days):
                pred_input = current_sequence.reshape(1, self.sequence_length, -1)
                pred_scaled = self.model.predict(pred_input, verbose=0)[0, 0]
                pred_price = self.target_scaler.inverse_transform([[pred_scaled]])[0, 0]
                predictions.append(pred_price)
                
                current_date += timedelta(days=1)
                while current_date.weekday() >= 5:
                    current_date += timedelta(days=1)
                
                current_sequence = self._update_features_for_prediction(
                    current_sequence, pred_price, current_date
                )
            
            mean_predictions = predictions
        
        # Generar fechas de predicción
        prediction_dates = []
        current_date = last_date
        for _ in range(trading_days):
            current_date += timedelta(days=1)
            while current_date.weekday() >= 5:
                current_date += timedelta(days=1)
            prediction_dates.append(current_date)
        
        # Crear DataFrame de resultados
        results = pd.DataFrame({
            'Date': prediction_dates,
            'Predicted_Price': mean_predictions
        })
        
        if confidence_intervals:
            results['Lower_95'] = lower_bound
            results['Upper_95'] = upper_bound
            results['Uncertainty'] = upper_bound - lower_bound
        
        return results

In [ ]:
# Crear predictor
predictor = FuturePredictor(
    model=best_model,
    feature_scaler=feature_scaler,
    target_scaler=target_scaler,
    feature_columns=FEATURE_COLUMNS,
    sequence_length=CONFIG['sequence_length']
)

print("✅ Predictor de futuro inicializado")

## 10.1 Predicción por Días

In [ ]:
# Predecir los próximos 30 días
predictions_30d = predictor.predict_future(
    last_data=df_clean,
    days=30,
    confidence_intervals=True,
    n_simulations=50
)

print("\n📅 Predicción para los próximos 30 días:")
predictions_30d

In [ ]:
# Visualizar predicción de 30 días
fig, ax = plt.subplots(figsize=(14, 6))

# Últimos 60 días de datos reales
recent_data = df_clean.tail(60)
ax.plot(recent_data['Date'], recent_data['Close'], 'b-', linewidth=2, label='Precio Histórico')

# Predicciones
ax.plot(predictions_30d['Date'], predictions_30d['Predicted_Price'], 'r-', linewidth=2, label='Predicción')
ax.fill_between(
    predictions_30d['Date'],
    predictions_30d['Lower_95'],
    predictions_30d['Upper_95'],
    alpha=0.3,
    color='red',
    label='Intervalo de Confianza 90%'
)

# Línea de separación
ax.axvline(x=df_clean['Date'].iloc[-1], color='gray', linestyle='--', alpha=0.7)
ax.text(df_clean['Date'].iloc[-1], ax.get_ylim()[1]*0.95, ' Hoy', fontsize=10)

ax.set_title(f'{CONFIG["ticker"]} - Predicción 30 Días', fontsize=14)
ax.set_xlabel('Fecha')
ax.set_ylabel('Precio ($)')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10.2 Predicción por Meses

In [ ]:
# Predecir los próximos 3 meses
predictions_3m = predictor.predict_future(
    last_data=df_clean,
    months=3,
    confidence_intervals=True,
    n_simulations=50
)

print(f"\n📅 Predicción para los próximos 3 meses ({len(predictions_3m)} días de trading):")
print(predictions_3m.head(10))
print("...")
print(predictions_3m.tail(10))

In [ ]:
# Visualizar predicción de 3 meses
fig, ax = plt.subplots(figsize=(14, 6))

# Últimos 90 días de datos reales
recent_data = df_clean.tail(90)
ax.plot(recent_data['Date'], recent_data['Close'], 'b-', linewidth=2, label='Precio Histórico')

# Predicciones
ax.plot(predictions_3m['Date'], predictions_3m['Predicted_Price'], 'r-', linewidth=2, label='Predicción')
ax.fill_between(
    predictions_3m['Date'],
    predictions_3m['Lower_95'],
    predictions_3m['Upper_95'],
    alpha=0.3,
    color='red',
    label='Intervalo de Confianza 90%'
)

ax.axvline(x=df_clean['Date'].iloc[-1], color='gray', linestyle='--', alpha=0.7)

ax.set_title(f'{CONFIG["ticker"]} - Predicción 3 Meses', fontsize=14)
ax.set_xlabel('Fecha')
ax.set_ylabel('Precio ($)')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10.3 Predicción hasta una Fecha Específica

In [ ]:
# Predecir hasta una fecha específica (ejemplo: 1 año desde hoy)
target_date = datetime.now() + timedelta(days=365)

predictions_to_date = predictor.predict_future(
    last_data=df_clean,
    target_date=target_date,
    confidence_intervals=True,
    n_simulations=30
)

print(f"\n🎯 Predicción hasta {target_date.strftime('%Y-%m-%d')}:")
print(f"   Días de trading predichos: {len(predictions_to_date)}")
print(f"\n   Precio final predicho: ${predictions_to_date['Predicted_Price'].iloc[-1]:.2f}")
print(f"   Rango (90% confianza): ${predictions_to_date['Lower_95'].iloc[-1]:.2f} - ${predictions_to_date['Upper_95'].iloc[-1]:.2f}")

In [ ]:
# Visualización completa con predicción de 1 año
fig, ax = plt.subplots(figsize=(16, 7))

# Datos históricos completos del último año
historical_data = df_clean.tail(252)
ax.plot(historical_data['Date'], historical_data['Close'], 'b-', linewidth=1.5, label='Precio Histórico')

# Predicciones
ax.plot(predictions_to_date['Date'], predictions_to_date['Predicted_Price'], 'r-', linewidth=1.5, label='Predicción')
ax.fill_between(
    predictions_to_date['Date'],
    predictions_to_date['Lower_95'],
    predictions_to_date['Upper_95'],
    alpha=0.2,
    color='red',
    label='Intervalo de Confianza 90%'
)

ax.axvline(x=df_clean['Date'].iloc[-1], color='green', linestyle='--', alpha=0.7, linewidth=2)
ax.text(df_clean['Date'].iloc[-1], ax.get_ylim()[1]*0.98, ' Hoy', fontsize=12, color='green', fontweight='bold')

ax.set_title(f'{CONFIG["ticker"]} - Predicción hasta {target_date.strftime("%Y-%m-%d")}', fontsize=14)
ax.set_xlabel('Fecha')
ax.set_ylabel('Precio ($)')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 11. 💾 Guardar Modelo y Artefactos

In [ ]:
import joblib
import json

# Guardar modelo
model.save('stock_prediction_advanced.keras')

# Guardar scalers
joblib.dump(feature_scaler, 'feature_scaler.pkl')
joblib.dump(target_scaler, 'target_scaler.pkl')

# Guardar configuración
config_to_save = CONFIG.copy()
config_to_save['feature_columns'] = FEATURE_COLUMNS
config_to_save['target_column'] = TARGET_COLUMN

with open('model_config.json', 'w') as f:
    json.dump(config_to_save, f, indent=2)

print("✅ Modelo y artefactos guardados:")
print("   - stock_prediction_advanced.keras")
print("   - feature_scaler.pkl")
print("   - target_scaler.pkl")
print("   - model_config.json")

# 12. 🔧 Función de Predicción para Producción

In [ ]:
def predict_stock_price(ticker, days=None, weeks=None, months=None, years=None, target_date=None):
    """
    Función completa para predecir precios de acciones.
    
    Args:
        ticker: Símbolo de la acción (ej: 'AAPL', 'GOOGL', 'MSFT')
        days: Días a predecir
        weeks: Semanas a predecir
        months: Meses a predecir
        years: Años a predecir
        target_date: Fecha específica (datetime o string 'YYYY-MM-DD')
        
    Returns:
        DataFrame con predicciones
    """
    import joblib
    import json
    from tensorflow.keras.models import load_model
    
    # Cargar configuración
    with open('model_config.json', 'r') as f:
        config = json.load(f)
    
    # Cargar modelo y scalers
    model = load_model('stock_prediction_advanced.keras', custom_objects={'AttentionLayer': AttentionLayer})
    feature_scaler = joblib.load('feature_scaler.pkl')
    target_scaler = joblib.load('target_scaler.pkl')
    
    # Descargar datos recientes
    now = datetime.now()
    start = datetime(now.year - 2, now.month, now.day)  # 2 años de historia
    df = yf.download(ticker, start, now, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df.reset_index()
    
    # Calcular indicadores
    df_features = calculate_technical_indicators(df)
    df_clean = df_features.dropna().reset_index(drop=True)
    
    # Crear predictor
    predictor = FuturePredictor(
        model=model,
        feature_scaler=feature_scaler,
        target_scaler=target_scaler,
        feature_columns=config['feature_columns'],
        sequence_length=config['sequence_length']
    )
    
    # Predecir
    if isinstance(target_date, str):
        target_date = datetime.strptime(target_date, '%Y-%m-%d')
    
    predictions = predictor.predict_future(
        last_data=df_clean,
        days=days,
        weeks=weeks,
        months=months,
        years=years,
        target_date=target_date,
        confidence_intervals=True
    )
    
    return predictions

print("✅ Función predict_stock_price() lista para usar")

## Ejemplo de Uso

In [ ]:
# Ejemplo: Predecir AAPL para los próximos 2 meses
# predictions = predict_stock_price('AAPL', months=2)
# predictions.head(20)

# Ejemplo: Predecir hasta una fecha específica
# predictions = predict_stock_price('AAPL', target_date='2025-12-31')
# predictions.tail()

# 13. 📋 Resumen del Modelo

## Mejoras Implementadas:

| Aspecto | Modelo Original | Modelo Mejorado |
|---------|----------------|------------------|
| **Features** | Solo Close (1) | 28 features técnicos |
| **Arquitectura** | LSTM básico | Bidirectional LSTM + Multi-Head Attention |
| **Regularización** | Ninguna | Dropout + BatchNorm + L2 |
| **Loss Function** | MSE | Huber (robusto a outliers) |
| **Callbacks** | Ninguno | Early Stopping + ReduceLR + Checkpointing |
| **Validación** | Train/Test | Train/Val/Test |
| **Predicción** | Solo siguiente día | Multi-horizonte (días/meses/años) |
| **Incertidumbre** | No | Intervalos de confianza |
| **Métricas** | MSE, R² | MSE, RMSE, MAE, MAPE, R², Direction Accuracy |

## Features Técnicos Incluidos:
- OHLCV (Open, High, Low, Close, Volume)
- Medias Móviles (SMA, EMA: 7, 14, 21, 50, 100, 200 días)
- RSI (Relative Strength Index)
- MACD (Moving Average Convergence Divergence)
- Bollinger Bands
- ATR (Average True Range)
- Volatilidad histórica
- Momentum y ROC
- Stochastic Oscillator
- Features temporales (día, mes, trimestre)

In [ ]:
print("\n" + "="*60)
print("🎉 NOTEBOOK COMPLETADO EXITOSAMENTE")
print("="*60)
print(f"\n📊 Ticker analizado: {CONFIG['ticker']}")
print(f"📈 Datos utilizados: {len(df_clean)} registros")
print(f"🧠 Arquitectura: Bidirectional LSTM + Multi-Head Attention")
print(f"📐 Features: {len(FEATURE_COLUMNS)} indicadores técnicos")
print(f"\n⚡ Métricas finales:")
for key, value in metrics.items():
    print(f"   {key}: {value:.4f}")
print("\n✅ Modelo listo para predicciones futuras")